# Census Age Data Cleaning — Niagara

## Statistics Canada Census Age Dataset

This notebook documents the cleaning and validation of the Statistics Canada Census Age dataset for Niagara Census Division and its 12 municipalities.

### Raw Dataset
`98100022.csv`

### Cleaned Dataset
`census_age_niagara_cleaned.csv`

### Census Years
- 2016
- 2021

### Main Tasks
- Validate the raw dataset
- Select the correct Census years
- Filter Niagara geography
- Validate DGUID
- Review and preserve the age hierarchy
- Preserve suppression and status information
- Validate missing values
- Validate duplicates
- Validate Niagara geographies
- Save the cleaned dataset

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("data/raw/98100022.csv")

print("Raw dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

## 1. Initial Validation

The raw dataset was inspected before any filtering or cleaning.

The initial validation checks the dataset dimensions, Census years, and geographic coverage.

In [ ]:
print("Dataset shape:", df.shape)

print("\nCensus years:")
print(df['Census year (2)'].value_counts().sort_index())

print("\nNumber of unique geographies:")
print(df['GEO'].nunique())

print("\nNumber of unique DGUIDs:")
print(df['DGUID'].nunique())

In [ ]:
# Check missing values and duplicates before cleaning
print("Raw missing values:", df.isna().sum().sum())
print("Raw exact duplicates:", df.duplicated().sum())

raw_keys = [
    'DGUID',
    'Census year (2)',
    'Age (in single years), average age and median age (128)'
]

print(
    "Raw duplicate DGUID-year-age records:",
    df.duplicated(subset=raw_keys).sum()
)

## 2. Prepare Niagara Geography

The analysis is restricted to Niagara Census Division and its 12 municipalities.

DGUID is retained as the reliable geographic identifier.

In [ ]:
# Step 5 — Prepare the Niagara Geography

niagara_dguids = [
    '2021A00033526',      # Niagara Census Division
    '2021A00053526003',   # Fort Erie
    '2021A00053526011',   # Port Colborne
    '2021A00053526014',   # Wainfleet
    '2021A00053526021',   # West Lincoln
    '2021A00053526028',   # Pelham
    '2021A00053526032',   # Welland
    '2021A00053526037',   # Thorold
    '2021A00053526043',   # Niagara Falls
    '2021A00053526047',   # Niagara-on-the-Lake
    '2021A00053526053',   # St. Catharines
    '2021A00053526057',   # Lincoln
    '2021A00053526065'    # Grimsby
]

niagara_df = df[df['DGUID'].isin(niagara_dguids)].copy()

# Clean whitespace from the original age column
age_col = 'Age (in single years), average age and median age (128)'
niagara_df[age_col] = niagara_df[age_col].str.strip()

# Validate the geography filter
print("Shape:", niagara_df.shape)
print("Unique DGUIDs:", niagara_df['DGUID'].nunique())
print("Unique geographies:", niagara_df['GEO'].nunique())
print(niagara_df[['GEO', 'DGUID']].drop_duplicates().sort_values('GEO').to_string(index=False))

In [ ]:
age_col = 'Age (in single years), average age and median age (128)'

print(
    "Age labels with leading/trailing whitespace:",
    niagara_df[age_col].ne(niagara_df[age_col].str.strip()).sum()
)

## 3. Census-Year Validation

The Census Age dataset contains two Census periods: 2016 and 2021.

Both Census years are retained. No annual values are created, interpolated, or forecasted.

In [ ]:
print("Census years in Niagara dataset:")
print(niagara_df['Census year (2)'].value_counts().sort_index())

## 4. Age-Category Review

The age variable contains 128 labels representing different levels of age information.

These include single-year ages, age groups, broad age groups, total age, average age, median age, and other nested categories.

The categories overlap and are therefore not summed together.

In [ ]:
age_col = 'Age (in single years), average age and median age (128)'

print("Number of unique age labels:")
print(niagara_df[age_col].nunique())

print("\nAge labels:")
print(niagara_df[age_col].unique())


In [ ]:
def classify_age_category(age):
    age = str(age).strip()

    if age == 'Average age':
        return 'Average age'
    if age == 'Median age':
        return 'Median age'
    if age == 'Total - Age':
        return 'Total age'
    if age == 'Under 1 year':
        return 'Under 1 year'
    if age.isdigit():
        return 'Single-year age'

    broad_groups = [
        '0 to 14 years',
        '15 to 64 years',
        '65 years and over',
        '85 years and over',
        '100 years and over'
    ]

    if age in broad_groups:
        return 'Broad age group'

    if ' to ' in age and age.endswith(' years'):
        return 'Age group'

    return 'Other'

In [ ]:
age_category_check = niagara_df[age_col].apply(classify_age_category)

print(age_category_check.value_counts())

## 5. Suppression and Status-Code Validation

Statistics Canada uses status symbols to identify observations requiring special treatment.

The `x` status represents suppressed observations. The `...` status is also preserved as provided by Statistics Canada.

Status-coded observations are not replaced with zero, estimated, or imputed values.

The numeric values and corresponding Symbol fields are retained.

In [ ]:
# Check gender status symbols in the Niagara dataset

symbol_cols = [
    'Symbol',
    'Symbol.1',
    'Symbol.2'
]

for col in symbol_cols:
    print(f"\n{col}")
    print(niagara_df[col].value_counts(dropna=False))

In [ ]:
for col in symbol_cols:
    print(f"{col}:")
    print("x =", (niagara_df[col] == 'x').sum())
    print("... =", (niagara_df[col] == '...').sum())

## 6. Gender Field Validation

The dataset contains three gender measures:

- Total-Gender
- Men+
- Women+

The original numeric values were preserved.

No attempt was made to force Men+ and Women+ to sum exactly to Total-Gender because Statistics Canada applies random rounding and confidentiality treatment.

In [ ]:
gender_cols = [
    'Gender (3a):Total - Gender[1]',
    'Gender (3a):Men+[2]',
    'Gender (3a):Women+[3]'
]

for col in gender_cols:
    print(f"\n{col}")
    print("Data type:", niagara_df[col].dtype)
    print("Missing values:", niagara_df[col].isna().sum())
    print("Numeric values:", pd.to_numeric(
        niagara_df[col], errors='coerce'
    ).notna().sum())

## 7. Missing-Value Validation

Missing values were checked after filtering to Niagara.

Blank Symbol fields are treated as the absence of a status symbol and are not interpreted as missing population values.

In [ ]:
check_cols = [
    'GEO',
    'DGUID',
    'Census year (2)',
    age_col,
    'Gender (3a):Total - Gender[1]',
    'Gender (3a):Men+[2]',
    'Gender (3a):Women+[3]',
    'Symbol',
    'Symbol.1',
    'Symbol.2'
]

print("Missing-value check:\n")

for col in check_cols:
    print(
        f"{col}: "
        f"{niagara_df[col].isna().sum()} missing "
        f"out of {len(niagara_df)}"
    )

In [ ]:
# Step 12 — Duplicate Validation

print("Exact duplicate rows:", niagara_df.duplicated().sum())

key_cols = [
    'DGUID',
    'Census year (2)',
    age_col
]

print(
    "Duplicate DGUID + Census year + Age records:",
    niagara_df.duplicated(subset=key_cols).sum()
)

In [ ]:
# Step 13 — Geographic Validation

expected_geographies = [
    'Niagara',
    'Fort Erie',
    'Port Colborne',
    'Wainfleet',
    'West Lincoln',
    'Pelham',
    'Welland',
    'Thorold',
    'Niagara Falls',
    'Niagara-on-the-Lake',
    'St. Catharines',
    'Lincoln',
    'Grimsby'
]

actual_geographies = sorted(niagara_df['GEO'].unique())

print("Number of geography levels:", len(actual_geographies))
print("\nGeographies:")
print(actual_geographies)

print("\nAll expected geographies present:",
      set(expected_geographies) == set(actual_geographies))

In [ ]:
# Step 14 — Validate Niagara Census Totals

total_age = niagara_df[
    niagara_df[age_col] == 'Total - Age'
]

print(
    total_age[
        ['Census year (2)', 'Gender (3a):Total - Gender[1]']
    ].sort_values('Census year (2)')
)

In [ ]:
# Compare municipality totals with Niagara regional total
population_col = 'Gender (3a):Total - Gender[1]'
year_col = 'Census year (2)'

for year in [2016, 2021]:
    year_data = total_age[total_age[year_col] == year]

    regional_total = year_data.loc[
        year_data['GEO'] == 'Niagara', population_col
    ].sum()

    municipalities = year_data[year_data['GEO'] != 'Niagara']
    municipality_total = municipalities[population_col].sum()

    difference = municipality_total - regional_total

    print(f"\nCensus year: {year}")
    print("Municipalities counted:", municipalities['GEO'].nunique())
    print("Municipality total:", municipality_total)
    print("Niagara regional total:", regional_total)
    print("Difference:", difference)

In [ ]:
from pathlib import Path

output_path = Path("census_age_niagara_cleaned.csv")

niagara_df.to_csv(output_path, index=False)

print("Cleaned dataset saved as:", output_path)
print("Final shape:", niagara_df.shape)

In [ ]:
# Final validation of saved dataset

final_df = pd.read_csv(output_path)

print("Saved dataset shape:", final_df.shape)
print("Exact duplicates:", final_df.duplicated().sum())

print("\nGeography count:")
print(final_df['GEO'].value_counts().sort_index())

print("\nCensus years:")
print(final_df['Census year (2)'].value_counts().sort_index())

In [ ]:
# Final validation of cleaned Niagara dataset

age_col = 'Age (in single years), average age and median age (128)'

print("Final shape:", niagara_df.shape)

print(
    "Age labels with whitespace:",
    niagara_df[age_col].ne(niagara_df[age_col].str.strip()).sum()
)

print("Unique DGUIDs:", niagara_df['DGUID'].nunique())

print("Exact duplicate rows:", niagara_df.duplicated().sum())

print(
    "Duplicate DGUID-year-age combinations:",
    niagara_df.duplicated(
        subset=['DGUID', 'Census year (2)', age_col]
    ).sum()
)

print("\nPopulation totals:")
print(
    niagara_df[
        niagara_df[age_col] == 'Total - Age'
    ].groupby(['GEO', 'Census year (2)'])[
        'Gender (3a):Total - Gender[1]'
    ].first()
)